# IOAI — 2026 Home Task Animal Deduction (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os
os.makedirs('data', exist_ok=True)
BASE='https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2026-home-task-animal-deduction'
for f in ['interactor.py','evaluate.py','animals_pool.txt','questions_pool.txt','dev.csv','test1.csv']:
    if not os.path.exists(f'data/{f}'): os.system(f'wget -q {BASE}/{f} -O data/{f}')
print('데이터 준비:', sorted(os.listdir('data')))
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Animal Deduction — 모범답안 (적응형 정보이득)

**핵심 아이디어**: 오라클은 결정적(같은 (동물,질문)→같은 답)이므로, 오프라인에서 `animals_pool × questions_pool` 답표를 한 번 계산해 둔다(자기 소유 Interactor 로). 추리 시에는 남은 후보를 **가장 균등하게 둘로 가르는 질문**(정보이득 최대, 예/아니오 비율이 50:50 에 가까운 질문)을 골라 물어 후보를 반씩 줄이고, 후보가 하나로 좁혀지면 guess 한다. 15문 예산 안에서 log2(1472)≈11 문이면 충분 → 높은 mean_score.

> 답표 사전계산은 오라클 호출이 많아 처음 1회가 느리다(수 분). 풀 크기를 줄이면 빨라진다.

In [ ]:
!pip install -q transformers accelerate
import os, sys
sys.path.insert(0, 'data'); os.chdir('data')
from evaluate import evaluate, load_pools
from interactor import Interactor
animals_pool, questions_pool = load_pools()
print(len(animals_pool), 'animals |', len(questions_pool), 'questions')

In [ ]:
import numpy as np, random

class MySolution:
    """적응형 정보이득 추리: 오프라인 답표 + 최대분할 질문 선택."""
    def __init__(self, animals_pool, questions_pool, n_questions=120, max_animals=None):
        self.animals = list(animals_pool) if max_animals is None else list(animals_pool)[:max_animals]
        # 질문은 정보량이 큰 것 위주로 일부만 사용(속도) — 무작위 표본
        qs = list(questions_pool); random.Random(0).shuffle(qs)
        self.questions = qs[:n_questions]
        print(f"[precompute] {len(self.animals)} x {len(self.questions)} answer table ...")
        self.table = np.zeros((len(self.animals), len(self.questions)), dtype=np.int8)
        for i, a in enumerate(self.animals):
            sim = Interactor(gold_animal=a, animals_pool=set(animals_pool),
                             questions_pool=set(questions_pool), budget=10**9)
            self.table[i] = [1 if sim.ask(q) == "yes" else 0 for q in self.questions]
            if (i + 1) % 300 == 0: print(f"  {i+1}/{len(self.animals)}")
        print("[precompute] done.")

    def solve(self, interactor):
        cand = np.arange(len(self.animals))          # 남은 후보 인덱스
        used = set()
        while not interactor.is_done():
            if len(cand) <= 1:
                interactor.guess(self.animals[cand[0]] if len(cand) else self.animals[0]); continue
            # 후보를 가장 균등하게 가르는(정보이득 최대) 질문 선택
            best_q, best_bal = None, -1
            for qi in range(len(self.questions)):
                if qi in used: continue
                yes = int(self.table[cand, qi].sum())
                bal = min(yes, len(cand) - yes)       # 50:50 에 가까울수록 큼
                if bal > best_bal: best_bal, best_q = bal, qi
            if best_q is None or best_bal == 0:
                interactor.guess(self.animals[cand[0]]); continue
            used.add(best_q)
            ans = 1 if interactor.ask(self.questions[best_q]) == "yes" else 0
            cand = cand[self.table[cand, best_q] == ans]
            if len(cand) == 0:                        # 답표 불일치 시 안전장치
                interactor.guess(self.animals[0]); break

# 자가 채점 (dev 150마리). 처음 사전계산이 오래 걸린다.
sol = MySolution(animals_pool, questions_pool)
res = evaluate(sol, "dev.csv")
print("dev mean_score:", round(res["mean_score"], 4), "| solved_rate:", round(res["solved_rate"], 4))

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv', 'submission.zip', 'submission.jsonl', 'submission.json']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)